# Phase 13 — Callaway–Sant'Anna

Re-estimates the exit effect with a staggered-adoption estimator.

**Input:** `data/interim/phase11_panel_controls.csv`

**Outputs**

| File | Contents |
|---|---|
| `data/results/phase13_callaway_santanna.csv` | event-study aggregation, by arm |
| `data/results/phase13_cohort_atts.csv` | the underlying ATT(g,t), by cohort |

For each cohort *g* and period *t*:

```
ATT(g,t) = [cohort g at t - cohort g at its reference] - [never treated at t - never treated at that reference]
```

Only never-retracted authors serve as controls. A single base period (-1) is
used, since each ATT is a difference of four group means. Standard errors come
from an author-level bootstrap and are not comparable with the journal-clustered
errors of Phase 12.

In [1]:
import os
import math
import numpy as np
import pandas as pd

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

PANEL = "data/interim/phase11_panel_controls.csv"
OUT_EVENT = "data/results/phase13_callaway_santanna.csv"
OUT_COHORT = "data/results/phase13_cohort_atts.csv"

ARMS = ["AUTHOR_MISCONDUCT", "HONEST_ERROR", "EDITORIAL_COMPROMISE"]
REF_ARM = "CONTROL"
OUTCOME = "active"

ANALYSIS_PRE, ANALYSIS_POST = 6, 6
BASE_EVENT = -1
EVENT_TIMES = list(range(-ANALYSIS_PRE, ANALYSIS_POST + 1))
HORIZON = 6
# Overall post-treatment average runs from +1; event time 0 is excluded
# because the retraction year mixes pre- and post-notice publishing.
POST_START = 1

N_BOOT = 200
BOOT_SEED = 20260101

COMPARE_TWFE = "data/results/phase12_results.csv"

pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 300)
os.makedirs("data/results", exist_ok=True)

print(f"outcome           {OUTCOME}")
print(f"comparison group  never-treated")
print(f"base period       event time {BASE_EVENT}")
print(f"post-period       +{POST_START} onward (event time 0 excluded)")
print(f"bootstrap         {N_BOOT} replications, clustered on author")

outcome           active
comparison group  never-treated
base period       event time -1
post-period       +1 onward (event time 0 excluded)
bootstrap         200 replications, clustered on author


## Load

In [2]:
p = pd.read_csv(PANEL, low_memory=False)
p = p[(p.event_time >= -ANALYSIS_PRE) & (p.event_time <= ANALYSIS_POST)]
p = p[p.arm.isin(ARMS + [REF_ARM])].copy()
p["cohort"] = np.where(p.arm == REF_ARM, 0, p.first_retraction_year).astype(int)

print(f"panel   {len(p):,} rows, {p.author_id.nunique():,} authors")
print(p.drop_duplicates("author_id").arm.value_counts().to_string())

coh = (p[p.cohort > 0].drop_duplicates("author_id")
         .groupby(["arm", "cohort"]).size().unstack(fill_value=0))
print("\ntreated authors by cohort")
print(coh.to_string())

panel   475,189 rows, 36,553 authors
arm
CONTROL                 19259
AUTHOR_MISCONDUCT        9617
HONEST_ERROR             4894
EDITORIAL_COMPROMISE     2783

treated authors by cohort
cohort                2015  2016  2017  2018  2019
arm                                               
AUTHOR_MISCONDUCT     1757  1861  1632  1852  2515
EDITORIAL_COMPROMISE   653   417   672   390   651
HONEST_ERROR           648   853   905  1222  1266


## Group-time effects

The panel is pivoted once into an author-by-year matrix, so the pivot is not
rebuilt inside each bootstrap draw.

In [3]:
W = p.pivot_table(index="author_id", columns="year", values=OUTCOME,
                  aggfunc="mean")
META = (p.drop_duplicates("author_id").set_index("author_id")[["cohort", "arm"]]
          .reindex(W.index))
VALS = W.to_numpy(dtype=float)          # NaN where unobserved
COHORT = META.cohort.to_numpy()
ARM = META.arm.to_numpy()
YCOL = {y: i for i, y in enumerate(W.columns)}

print(f"wide matrix: {W.shape[0]:,} authors x {W.shape[1]} years")


def att_gt(g, t, base_year, arm, rows=None):
    """One group-time average treatment effect.

    Authors enter only if observed in both periods, so each side is a
    within-author change. `rows`, when given, is a positional index array for a
    bootstrap resample.
    """
    if t not in YCOL or base_year not in YCOL:
        return np.nan, 0, 0
    delta = VALS[:, YCOL[t]] - VALS[:, YCOL[base_year]]
    ok = ~np.isnan(delta)

    m_treat = ok & (COHORT == g) & (ARM == arm)
    m_ctrl = ok & (COHORT == 0)

    if rows is not None:
        w = np.bincount(rows, minlength=len(delta)).astype(float)
        nt, nc = w[m_treat].sum(), w[m_ctrl].sum()
        if nt == 0 or nc == 0:
            return np.nan, int(nt), int(nc)
        dt = float(np.dot(delta[m_treat], w[m_treat]) / nt)
        dc = float(np.dot(delta[m_ctrl], w[m_ctrl]) / nc)
        return dt - dc, int(nt), int(nc)

    nt, nc = int(m_treat.sum()), int(m_ctrl.sum())
    if nt == 0 or nc == 0:
        return np.nan, nt, nc
    return float(delta[m_treat].mean() - delta[m_ctrl].mean()), nt, nc


def att_table(arm, rows=None):
    """Every cohort at every period that cohort can reach."""
    cohorts = sorted(int(c) for c in np.unique(COHORT[ARM == arm]) if c > 0)
    out = []
    for g in cohorts:
        base = g + BASE_EVENT
        for e in EVENT_TIMES:
            if e == BASE_EVENT:
                continue
            a, nt, nc = att_gt(g, g + e, base, arm, rows=rows)
            if not np.isfinite(a) or nt == 0:
                continue
            out.append({"arm": arm, "cohort": g, "year": g + e,
                        "event_time": e, "att": a,
                        "n_treated": nt, "n_control": nc})
    return pd.DataFrame(out)

wide matrix: 36,553 authors x 17 years


## Aggregation and inference

Cohort effects are averaged at each event time with weights proportional to
cohort size. Weights are non-negative and sum to one within each event time.

In [4]:
def aggregate_event(att):
    if att.empty:
        return pd.DataFrame()
    rows = []
    for e, sub in att.groupby("event_time"):
        w = sub.n_treated.astype(float)
        if w.sum() <= 0:
            continue
        rows.append({"event_time": e,
                     "att": float(np.average(sub.att, weights=w)),
                     "n_cohorts": int(sub.cohort.nunique()),
                     "n_treated": int(sub.n_treated.sum())})
    return pd.DataFrame(rows).sort_values("event_time").reset_index(drop=True)


def aggregate_overall(att, min_event=POST_START):
    post = att[att.event_time >= min_event]
    if post.empty:
        return np.nan
    return float(np.average(post.att, weights=post.n_treated.astype(float)))


def bootstrap(arm, n_boot=N_BOOT, seed=BOOT_SEED):
    """Author-level resampling, preserving within-author correlation."""
    rng = np.random.default_rng(seed)
    idx_t = np.flatnonzero((ARM == arm) & (COHORT > 0))
    idx_c = np.flatnonzero(COHORT == 0)

    ev_draws, overall = [], []
    for b in range(n_boot):
        rows = np.concatenate([
            rng.choice(idx_t, len(idx_t), replace=True),
            rng.choice(idx_c, len(idx_c), replace=True)])
        tab = att_table(arm, rows=rows)
        if tab.empty:
            continue
        ev = aggregate_event(tab)
        if not ev.empty:
            ev_draws.append(ev.set_index("event_time").att)
        overall.append(aggregate_overall(tab))

    ev_se = (pd.concat(ev_draws, axis=1).std(axis=1)
             if ev_draws else pd.Series(dtype=float))
    return ev_se, (float(np.nanstd(overall)) if overall else np.nan)


def _norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def pvalue(coef, se):
    if not (np.isfinite(coef) and np.isfinite(se)) or se <= 0:
        return np.nan
    return 2 * (1 - _norm_cdf(abs(coef / se)))

## Estimate

In [5]:
results, cohort_tables = {}, []

for arm in ARMS:
    att = att_table(arm)
    if att.empty:
        continue
    cohort_tables.append(att)

    ev = aggregate_event(att)
    overall = aggregate_overall(att)
    ev_se, overall_se = bootstrap(arm)

    ev["se"] = ev.event_time.map(ev_se)
    ev["p"] = [pvalue(c, s) for c, s in zip(ev.att, ev.se)]
    ev["ci_lo"] = ev.att - 1.96 * ev.se
    ev["ci_hi"] = ev.att + 1.96 * ev.se
    ev["arm"] = arm
    results[arm] = (ev, overall, overall_se)

    print(f"\n{arm}")
    show = ev[["event_time", "att", "se", "p", "ci_lo", "ci_hi",
               "n_treated"]].round(4)
    print(show.to_string(index=False))
    print(f"  overall post-treatment: {overall:+.4f} "
          f"(se {overall_se:.4f}, p {pvalue(overall, overall_se):.4f})")


AUTHOR_MISCONDUCT
 event_time     att     se      p   ci_lo   ci_hi  n_treated
         -6 -0.0512 0.0060 0.0000 -0.0629 -0.0395       9617
         -5 -0.0443 0.0056 0.0000 -0.0553 -0.0333       9617
         -4 -0.0322 0.0045 0.0000 -0.0410 -0.0234       9617
         -3 -0.0124 0.0047 0.0085 -0.0216 -0.0032       9617
         -2 -0.0240 0.0046 0.0000 -0.0329 -0.0151       9617
          0  0.0734 0.0044 0.0000  0.0648  0.0820       9617
          1 -0.0566 0.0045 0.0000 -0.0654 -0.0477       9617
          2 -0.0519 0.0049 0.0000 -0.0615 -0.0422       9617
          3 -0.0467 0.0049 0.0000 -0.0563 -0.0371       9617
          4 -0.0303 0.0053 0.0000 -0.0406 -0.0199       9617
          5 -0.0297 0.0053 0.0000 -0.0400 -0.0193       9617
          6 -0.0117 0.0054 0.0297 -0.0223 -0.0012       9617
  overall post-treatment: -0.0378 (se 0.0041, p 0.0000)

HONEST_ERROR
 event_time     att     se      p   ci_lo   ci_hi  n_treated
         -6 -0.0922 0.0078 0.0000 -0.1075 -0.0770       4

## Cohort heterogeneity

Cohorts are estimated separately. Both +3 and +6 are reported: a disagreement
confined to +6, where later cohorts reach the edge of the data, points to
incomplete coverage, while one present at both is a difference between cohorts.

In [6]:
cohort_summary = pd.DataFrame()
if cohort_tables:
    all_att = pd.concat(cohort_tables, ignore_index=True)

    for h in (3, HORIZON):
        sub = all_att[all_att.event_time == h]
        if sub.empty:
            continue
        print(f"\nATT at +{h}, by cohort")
        print(sub.pivot(index="cohort", columns="arm", values="att")
                 .round(4).to_string())

    summary = []
    for h in (3, HORIZON):
        sub = all_att[all_att.event_time == h]
        if sub.empty:
            continue
        for arm, s in sub.groupby("arm"):
            s = s.sort_values("cohort")
            slope = (float(np.polyfit(s.cohort, s.att, 1)[0])
                     if len(s) >= 2 else np.nan)
            summary.append({"event_time": h, "arm": arm,
                            "spread": round(float(s.att.max() - s.att.min()), 4),
                            "slope_per_cohort_year": round(slope, 4),
                            "earliest": round(float(s.att.iloc[0]), 4),
                            "latest": round(float(s.att.iloc[-1]), 4)})

    cohort_summary = pd.DataFrame(summary)
    if not cohort_summary.empty:
        print("\nspread and gradient by arm")
        print(cohort_summary.sort_values(["arm", "event_time"])
                            .to_string(index=False))


ATT at +3, by cohort
arm     AUTHOR_MISCONDUCT  EDITORIAL_COMPROMISE  HONEST_ERROR
cohort                                                       
2015              -0.0906               -0.1246       -0.0546
2016              -0.0680               -0.0556       -0.0735
2017              -0.0396               -0.0037       -0.0523
2018              -0.0442               -0.0051       -0.0338
2019              -0.0068               -0.0005       -0.0169

ATT at +6, by cohort
arm     AUTHOR_MISCONDUCT  EDITORIAL_COMPROMISE  HONEST_ERROR
cohort                                                       
2015              -0.0719               -0.0760       -0.0276
2016              -0.0349               -0.0418       -0.0610
2017               0.0108                0.0236       -0.0245
2018              -0.0011                0.0407        0.0055
2019               0.0250                0.0041        0.0110

spread and gradient by arm
 event_time                  arm  spread  slope_per_cohort_y

## Comparison with two-way fixed effects

In [7]:
comparison = pd.DataFrame()
twfe = None
if os.path.isfile(COMPARE_TWFE):
    t = pd.read_csv(COMPARE_TWFE)
    twfe = t[(t.outcome == OUTCOME) & (t.event_time == HORIZON)]

rows = []
for arm in ARMS:
    if arm not in results:
        continue
    ev, overall, overall_se = results[arm]
    at = ev[ev.event_time == HORIZON]
    row = {"arm": arm,
           "CS": round(float(at.att.iloc[0]), 4) if len(at) else np.nan,
           "CS_se": round(float(at.se.iloc[0]), 4) if len(at) else np.nan,
           "CS_overall_post": round(overall, 4)}
    if twfe is not None:
        m = twfe[twfe.group == arm]
        row["TWFE"] = round(float(m.coef.iloc[0]), 4) if len(m) else np.nan
    rows.append(row)

if rows:
    comparison = pd.DataFrame(rows)
    cols = ["arm"] + [c for c in ["TWFE", "CS", "CS_se", "CS_overall_post"]
                      if c in comparison.columns]
    print(comparison[cols].to_string(index=False))

    cs = comparison.CS.dropna()
    if len(cs) > 1:
        print(f"\nspread across arms at +{HORIZON}: {float(cs.max() - cs.min()):.4f}")
    if "CS_overall_post" in comparison.columns:
        op = comparison.CS_overall_post.dropna()
        if len(op) > 1:
            print(f"spread across arms, overall post: {float(op.max() - op.min()):.4f}")

                 arm    TWFE      CS  CS_se  CS_overall_post
   AUTHOR_MISCONDUCT -0.0286 -0.0117 0.0054          -0.0378
        HONEST_ERROR -0.0251 -0.0146 0.0070          -0.0351
EDITORIAL_COMPROMISE -0.0173 -0.0117 0.0083          -0.0342

spread across arms at +6: 0.0029
spread across arms, overall post: 0.0036


## Write

In [8]:
ev_all = [ev for ev, _, _ in results.values()]
if ev_all:
    out = pd.concat(ev_all, ignore_index=True)
    out.to_csv(OUT_EVENT, index=False)
    print(f"{OUT_EVENT}  {len(out):,} rows")

if cohort_tables:
    out_c = pd.concat(cohort_tables, ignore_index=True)
    out_c.to_csv(OUT_COHORT, index=False)
    print(f"{OUT_COHORT}  {len(out_c):,} group-time effects")

data/results/phase13_callaway_santanna.csv  36 rows
data/results/phase13_cohort_atts.csv  180 group-time effects


---

Pre-period estimates are produced by construction: event times below the base
period compare cohorts to never-treated authors before the event. Both
estimators are reported; Phase 12 is the conventional specification.